## voir_coquilles

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock du 21.05.2026) et `./data/coquilles_filtrees_20260630_084651.csv` (produit par `recensement_coquilles_filtres.ipynb`)

**Fichier(s) de sortie :** DataFrame en mémoire

**Description :** Vérification visuelle d'un échantillon de coquilles : tirage de 3 retraits et 3 transferts dans le CSV du recensement, extraction de leur texte depuis l'archive BOFiP, et affichage pour contrôle (avis de retrait/transfert vs. doctrine).

## Cellule 1 — Chemins

Seule cellule a modifier. Indiquer le chemin de l'archive et du CSV
produit par le recensement.

In [1]:
import os, re, tarfile, pandas as pd

TGZ = r"./data/bofip_stock_live_20260521.tgz"
CSV = r"./data/coquilles_filtrees_20260630_084651.csv"

def sans_balises(html):
    corps = html.split("<body>")[-1] if "<body>" in html else html
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", corps)).strip()

if os.path.isfile(TGZ):
    print(f"Archive : {os.path.basename(TGZ)}")
else:
    print("Archive non trouvee.")
if os.path.isfile(CSV):
    print(f"CSV : {os.path.basename(CSV)}")
else:
    print("CSV non trouve.")

Archive : bofip_stock_live_20260521.tgz
CSV : coquilles_filtrees_20260630_084651.csv


## Cellule 2 — Choisir 3 retraits et 3 transferts

On exclut les trois temoins deja verifies (1574, 3868, 1360) pour
tirer des documents qu'on n'a pas encore vus.

In [2]:
df = pd.read_csv(CSV, sep=';')
temoins = {"1574-PGP", "3868-PGP", "1360-PGP"}
retraits = df[(df['source'] == 'retrait') & (~df['identifiant'].isin(temoins))]
transferts = df[(df['source'] == 'transfert') & (~df['identifiant'].isin(temoins))]

echantillon = pd.concat([
    retraits.sample(3, random_state=42),
    transferts.sample(3, random_state=42),
])

print(f'Retraits tires   : {list(echantillon[echantillon["source"]=="retrait"]["identifiant"])}')
print(f'Transferts tires : {list(echantillon[echantillon["source"]=="transfert"]["identifiant"])}')

Retraits tires   : ['8324-PGP', '7240-PGP', '7854-PGP']
Transferts tires : ['6493-PGP', '4837-PGP', '522-PGP']


## Cellule 3 — Extraire le texte depuis l'archive

On ouvre l'archive une seule fois et on lit le `data.html` de chaque
document selectionne.

In [3]:
codes_voulus = set(echantillon['identifiant'])
textes = {}

with tarfile.open(TGZ, 'r:gz') as tar:
    for m in tar:
        if not m.isfile() or not m.name.endswith('data.html'):
            continue
        for c in codes_voulus:
            if f'/{c}/' in m.name.replace('\\', '/'):
                html = tar.extractfile(m).read().decode('utf-8', errors='replace')
                textes[c] = sans_balises(html)
                break

print(f'{len(textes)} documents extraits sur {len(codes_voulus)} attendus.')

6 documents extraits sur 6 attendus.


## Cellule 4 — Lire le contenu de chaque coquille

Chaque document est affiche avec son type (retrait ou transfert) et
sa longueur. Verifier que le texte est bien un avis de retrait ou de
transfert, et non de la doctrine.

In [4]:
for _, row in echantillon.iterrows():
    ident = row['identifiant']
    print('=' * 70)
    print(f"{ident}  |  type : {row['source']}  |  {row['longueur']} car.")
    print('=' * 70)
    if ident in textes:
        print(textes[ident])
    else:
        print('(non trouve dans l\'archive)')
    print()

8324-PGP  |  type : retrait  |  1326 car.
Dans une décision n° 2017-660 QPC du 6 octobre 2017 , le Conseil constitutionnel a jugé contraire à la Constitution le premier alinéa du paragraphe I de l' article 235 ter ZCA du code général des impôts (CGI) relatif à la contribution additionnelle à l'impôt sur les sociétés de 3 % au titre des montants distribués. La déclaration d'inconstitutionnalité de la contribution intervient à compter du 6 octobre 2017 et est applicable à toutes les affaires non jugées définitivement à cette date. En outre, l' article 37 de la loi n° 2017-1837 du 30 décembre 2017 de finances pour 2018 a supprimé la contribution additionnelle à l’impôt sur les sociétés de 3 % au titre des montants distribués, pour les montants distribués dont la mise en paiement intervient à compter du 1 er janvier 2018. Les commentaires exprimés dans le présent document sont retirés à compter de la date de publication de la présente version. Pour prendre connaissance des commentaires ant